In [ ]:
from datetime import date
from time import process_time, time
from imageio import imread
from rocketpy import Environment
import concurrent.futures
from Helpers.ConcurrentFunctions import RunFlightWithMonteCarlo
from IPython.display import display
#Calculate and plot landing elipses
# Import libraries
from matplotlib.patches import Ellipse
from Helpers import FlightParams
%config InlineBackend.figure_formats = ['svg']
import matplotlib as mpl
import matplotlib.pyplot as plt

%matplotlib inline
mpl.rcParams["figure.figsize"] = [8, 5]
mpl.rcParams["figure.dpi"] = 120
mpl.rcParams["font.size"] = 14
mpl.rcParams["legend.fontsize"] = 14
mpl.rcParams["figure.titlesize"] = 14


import logging
import numpy as np


resultsFile = open(FlightParams.Parameters["generatedFilesLocation"] + "results.txt", "w")

# envParams["file"] = row[0].value

print("TOTAL SIMS IS: " + str(FlightParams.Parameters["numberSims"]))
resultsFile.write("TOTAL SIMS IS: " + str(FlightParams.Parameters["numberSims"]) + "\n\n\n")


numIterations = 0
filename = f"MonteCarlo{numIterations}"
MonteCarloErrorFile = open("../Output/" + str(filename) + "_Errors.txt", "w")
MonteCarloInputFile = open("../Output/" + str(filename) + "_Input.txt", "w")
MonteCarloOutputFile = open("../Output/" + str(filename) + "_Output.txt", "w")

if(FlightParams.Parameters["doing_monte_carlo"]):
    allResults = []
    
    while True:
        theLaunchDate = [*FlightParams.Parameters["day"], FlightParams.Parameters["hour"]]

        printedString = "Current parameters are: "
        for i in range(len(FlightParams.Parameters["readable_names"])):
            printedString += FlightParams.Parameters["readable_names"][i] + ":"
            trueValue = FlightParams.Parameters["true_ranges"][i]
            trueValue = trueValue[FlightParams.Parameters["true_current_indices"][i]]
            printedString += str(trueValue) + " "

        if(len(FlightParams.Parameters["readable_names"]) > 0):
            print(printedString)
            resultsFile.write(printedString)

            if numIterations > 0:
                filename = f"MonteCarlo{numIterations}"
                MonteCarloErrorFile = open("../Output/" + str(filename) + "_Errors.txt", "w")
                MonteCarloInputFile = open("../Output/" + str(filename) + "_Input.txt", "w")
                MonteCarloOutputFile = open("../Output/" + str(filename) + "_Output.txt", "w")
        
        initial_wall_time = time()
        initial_cpu_time = process_time()
        out = display("Starting", display_id=True)
        number_of_simulations = FlightParams.Parameters["numberSims"]
        totalProcesses = FlightParams.Parameters["processes"]
        simsPerProcess = number_of_simulations / totalProcesses



        if __name__ == "__main__":
            with concurrent.futures.ProcessPoolExecutor(max_workers=totalProcesses) as executor:
                results = [executor.submit(RunFlightWithMonteCarlo, simsPerProcess, initial_cpu_time) for _ in range(totalProcesses)] 
                for result in concurrent.futures.as_completed(results):
                    MonteCarloInputFile.write(result.result()[0] + "\n")
                    MonteCarloOutputFile.write(result.result()[1] + "\n")
                    MonteCarloErrorFile.write(result.result()[2] + "\n")

        ## Print and save total time
        final_string = f"Completed {number_of_simulations} iterations successfully. Total CPU time: {process_time() - initial_cpu_time} s. Total wall time: {time() - initial_wall_time} s"
        resultsFile.write(f"Completed {number_of_simulations} iterations successfully. Total CPU time: {process_time() - initial_cpu_time} s. Total wall time: {time() - initial_wall_time} s\n")
        out.update(final_string)
        MonteCarloInputFile.write(final_string + "\n")
        MonteCarloOutputFile.write(final_string + "\n")
        MonteCarloErrorFile.write(final_string + "\n")


        # Initialize variable to store all results
        dispersion_general_results = []

        dispersion_results = {
            "out_of_rail_time": [],
            "out_of_rail_velocity": [],
            "apogee_time": [],
            "apogee_altitude": [],
            "apogee_x": [],
            "apogee_y": [],
            "impact_x": [],
            "impact_y": [],
            "initial_static_margin": [],
            "out_of_rail_static_margin": [],
            "out_of_rail_stability_margin": [],
            "max_velocity": [],
            "execution_time": [],
        }

        # Get all dispersion results
        # Get file
        dispersion_output_file = open("../Output/" + str(filename) + "_Output.txt", "r+")

        # Read each line of the file and convert to dict
        for line in dispersion_output_file:
            # Skip comments lines
            if line[0] != "{":
                continue
            # Eval results and store them
            flight_result = eval(line)
            dispersion_general_results.append(flight_result)
            for parameter_key, parameter_value in flight_result.items():
                dispersion_results[parameter_key].append(parameter_value)

        # Close data file
        dispersion_output_file.close()

        # Print number of flights simulated
        N = len(dispersion_general_results)
        print("Number of simulations: ", N)
        resultsFile.write("Number of simulations: " + str(N) + "\n")

        # Retrieve dispersion data por apogee and impact XY position
        apogee_x = np.array(dispersion_results["apogee_x"])
        apogee_y = np.array(dispersion_results["apogee_y"])

        FlightParams.createDistributionPlot(resultsFile, dispersion_results["out_of_rail_stability_margin"], "Out Of Rail Stability Margin", lowerProbBound = 1.5, upperProbBound = 4, maxDecimals = 4, percentConfidence = 90, hypothesis = None, unit = "none", makeGraph = True, filename = "Rail" + str(numIterations))
        print("\n\n")
        resultsFile.write("\n\n")
        FlightParams.createDistributionPlot(resultsFile, dispersion_results["apogee_altitude"], "Apogee Altitude", lowerProbBound = 9900 * 0.3048, upperProbBound = 10100 * 0.3048, maxDecimals = 0, percentConfidence = 90, hypothesis = 3048, unit = "meter", makeGraph = True, filename = "Apogee" + str(numIterations))
        print("\n\n\n----------------------------------------------------------------------------------------------------\n\n\n")
        resultsFile.write("\n\n\n----------------------------------------------------------------------------------------------------\n\n\n")

        configuration = []

        for i in range(len(FlightParams.Parameters["true_current_indices"])):
            trueIndexInRange = FlightParams.Parameters["true_current_indices"][i]
            trueVal = FlightParams.Parameters["true_ranges"][trueIndexInRange]
            if(FlightParams.Parameters["readable_names"][i] == "Day"):
                trueVal = (date(*trueVal) - FlightParams.Parameters["start_day"]).days
            configuration.append(trueVal)



        allResults.append((configuration, np.mean(dispersion_results["apogee_altitude"])))


        if len(dispersion_results["impact_x"]) > 0:
            img = imread("../InputFiles/background.jpg")
            # Retrieve dispersion data por apogee and impact XY position
            impact_x = np.array(dispersion_results["impact_x"])
            impact_y = np.array(dispersion_results["impact_y"])

            print(f"Simulated Landing Position x = {np.mean(dispersion_results['impact_x'])}, y = {np.mean(dispersion_results['impact_y'])}")
            resultsFile.write(f"Simulated Landing Position x = {np.mean(dispersion_results['impact_x'])}, y = {np.mean(dispersion_results['impact_y'])}\n")
            # # Define function to calculate eigen values
            def eigsorted(cov):
                vals, vecs = np.linalg.eigh(cov)
                order = vals.argsort()[::-1]
                return vals[order], vecs[:, order]


            # Create plot figure
            plt.figure(num=None, figsize=(8, 6), dpi=150, facecolor="w", edgecolor="k")
            ax = plt.subplot(111)

            # Calculate error ellipses for impact
            impactCov = np.cov(impact_x, impact_y)
            impactVals, impactVecs = eigsorted(impactCov)
            impactTheta = np.degrees(np.arctan2(*impactVecs[:, 0][::-1]))
            impactW, impactH = 2 * np.sqrt(impactVals)

            # Draw error ellipses for impact
            impact_ellipses = []
            for j in [1, 2, 3]:
                impactEll = Ellipse(
                    xy=(np.mean(impact_x), np.mean(impact_y)),
                    width=impactW * j,
                    height=impactH * j,
                    angle=impactTheta,
                    color="black",
                )
                impactEll.set_facecolor((0, 0, 1, 0.2))
                impact_ellipses.append(impactEll)
                ax.add_artist(impactEll)

            # Calculate error ellipses for apogee
            apogeeCov = np.cov(apogee_x, apogee_y)
            apogeeVals, apogeeVecs = eigsorted(apogeeCov)
            apogeeTheta = np.degrees(np.arctan2(*apogeeVecs[:, 0][::-1]))
            apogeeW, apogeeH = 2 * np.sqrt(apogeeVals)

            # Draw error ellipses for apogee
            for j in [1, 2, 3]:
                apogeeEll = Ellipse(
                    xy=(np.mean(apogee_x), np.mean(apogee_y)),
                    width=apogeeW * j,
                    height=apogeeH * j,
                    angle=apogeeTheta,
                    color="black",
                )
                apogeeEll.set_facecolor((0, 1, 0, 0.2))
                ax.add_artist(apogeeEll)

            # Draw launch point
            plt.scatter(0, 0, s=30, marker="*", color="black", label="Launch Point")
            # Draw apogee points
            plt.scatter(
                apogee_x, apogee_y, s=5, marker="^", color="green", label="Simulated Apogee"
            )
            # Draw impact points
            plt.scatter(
                impact_x, impact_y, s=5, marker="v", color="blue", label="Simulated Landing Point"
            )

            plt.legend()

            # Add title and labels to plot
            ax.set_title(
                "1$\sigma$, 2$\sigma$ and 3$\sigma$ Dispersion Ellipses: Apogee and Lading Points"
            )
            ax.set_ylabel("North (m)")
            ax.set_xlabel("East (m)")

            # Add background image to plot
            # You can translate the basemap by changing dx and dy (in meters)
            dx = 0
            dy = 0
            plt.imshow(img, zorder=0, extent=[-1000 - dx, 1000 - dx, -1000 - dy, 1000 - dy])

            plt.axhline(0, color="black", linewidth=0.5)
            plt.axvline(0, color="black", linewidth=0.5)
            plt.xlim(-5000, 5000)
            plt.ylim(-5000, 5000)

            # Save plot and show result
            plt.savefig("../Output/" + str(filename) + ".pdf", bbox_inches="tight", pad_inches=0)
            plt.savefig("../Output/" + str(filename) + ".svg", bbox_inches="tight", pad_inches=0)
            plt.show()
            print("\n\n\n----------------------------------------------------------------------------------------------------\n\n\n")
            resultsFile.write("\n\n\n----------------------------------------------------------------------------------------------------\n\n\n")

        nowTerminate = FlightParams.NextIteration() or len(FlightParams.Parameters["readable_names"]) > 0

        MonteCarloInputFile.close()
        MonteCarloOutputFile.close()
        MonteCarloErrorFile.close()

        if nowTerminate:
            resultsFile.close()
            break

    if len(FlightParams.Parameters["readable_names"]) > 0:
        print(allResults)

        plt.scatter(varyingParamVals, apogeeAtIteration)
        
        if(FlightParams.varyingVariable == "airbrake"):
            plt.title("Apogees at Different Deployment Times")
            plt.xlabel("Airbrake Deployment Time (Seconds after Burnout)")
            plt.ylabel("Apogee for that Time")
        elif(FlightParams.varyingVariable == "finposition"):
            plt.title("Apogees at Different Fin Positions")
            plt.xlabel("Fin Position")
            plt.ylabel("Apogee for that Position")
        elif(FlightParams.varyingVariable == "weatherHour"):
            plt.title(f"Apogees for Different Hours On Day {day}")
            plt.xlabel("Hour (Military Time)")
            plt.ylabel("Apogee at that hour")
            plt.axhline(3048, color='k', linestyle='solid', linewidth=4)
            plt.text(9.75, 3048, f'Desired Apogee', va='bottom', ha='left', color='red')
            
        #creating the trendline with the given x and y values
        trendline = np.polyfit(varyingParamVals, apogeeAtIteration, 1)
        trendline = np.poly1d(trendline)
        # obtaining the y values from the trendline
        y_vals = trendline(varyingParamVals)

        # R^2 calculation
        residuals = apogeeAtIteration - y_vals
        ss_res = np.sum(residuals**2)
        ss_tot = np.sum((apogeeAtIteration - np.mean(apogeeAtIteration))**2)
        r_squared = 1 - (ss_res / ss_tot)

        # plotting the trendline
        plt.plot(varyingParamVals, y_vals, "r", label="Trendline (R² = " + str(round(r_squared, 3)) + ")")
        plt.legend()

        plt.savefig(FlightParams.Parameters["generatedFilesLocation"] + f"FinalSimTable{day}.png", bbox_inches="tight", pad_inches=0)
        plt.show()
else:
    initial_wall_time = time()
    initial_cpu_time = process_time()
    flightData = RunFlightWithMonteCarlo(1, initial_cpu_time)
    MonteCarloInputFile.write(flightData[0] + "\n")
    MonteCarloOutputFile.write(flightData[1] + "\n")
    MonteCarloErrorFile.write(flightData[2] + "\n")

    ## Print and save total time
    final_string = f"Completed sim successfully. Total CPU time: {process_time() - initial_cpu_time} s. Total wall time: {time() - initial_wall_time} s"
    MonteCarloInputFile.write(final_string + "\n")
    MonteCarloOutputFile.write(final_string + "\n")
    MonteCarloErrorFile.write(final_string + "\n")

    MonteCarloInputFile.close()
    MonteCarloOutputFile.close()
    MonteCarloErrorFile.close()


<>:230: SyntaxWarning: invalid escape sequence '\s'
<>:230: SyntaxWarning: invalid escape sequence '\s'
C:\Users\mahd\AppData\Local\Temp\ipykernel_32336\347375037.py:230: SyntaxWarning: invalid escape sequence '\s'
  "1$\sigma$, 2$\sigma$ and 3$\sigma$ Dispersion Ellipses: Apogee and Lading Points"


YOU MUST STILL FILL IN THfE FOLLOWING VARIABLES:
VARIABLES START


start_day,
day,
hour,
doing_monte_carlo,
iterative_variables,
readable_names,
true_names,
true_ranges,
true_current_indices,
hasBottail,
obtaining_motor_csv,
numberSims,
processes,
latitude,
longitude,
elevation,
atmosphere_type,
env_file,
fahrenheit_temp,
pressure,
wind_u,
wind_v,
generatedFilesLocation,
dryMotorMass,
propellant_mass,
grainInnerRadius,
grainOuterRadius,
grainHeight,
the_nozzle_radius,
the_throat_radius,
the_nozzle_position,
grain_center_of_mass_position,
center_of_dry_mass_within_motor,
motor_thrust_file,
burn_time,
numGrains,
grainSeparation,
the_motor_position,
the_center_of_mass_without_motor,
rocket_center_of_dry_mass_position,
power_off_file,
power_on_file,
inclination,
heading,
rail_length,
using-airbrakes,
air_brake_drag,
airbrake_sample_rate,
airbrake_clamp,
override_rocketdrag_with_airbrakedrag,
airbrake_area,
terminate_at_apogee,


VARIABLES END


Impulse is: 9680.79964519286
TOTAL SIMS IS: 5

'Completed 5 iterations successfully. Total CPU time: 0.046875 s. Total wall time: 15.057209014892578 s'

Number of simulations:  0


NameError: name 'currentVal' is not defined